# Data Loading, Cleaning & Feature Engineering

Bike Share Toronto 2023 Ridership Analysis

This notebook loads the 12 monthly CSV files, cleans column inconsistencies, engineers temporal and categorical features, removes outliers using the IQR method, and exports a single processed CSV for downstream analysis.

All feature engineering logic lives in `src/features.py` so that every notebook works from the same definitions.

In [ ]:
import sys
from pathlib import Path

# Ensure that the project root is on the path so src/ can be imported
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
from src.load_data import load_bike_ridership_2023

df_raw = load_bike_ridership_2023()
print(f"Loaded {len(df_raw):,} rows")
df_raw.head()

Loaded 5,713,141 rows


,Trip Duration,Start Station Id,Start Time,Start Station Name,End Station Id,End Time,End Station Name,User Type
Trip Id,,,,,,,,
20148784,840,7022,01/01/2023 00:00,Simcoe St / Queen St W,7703.0,01/01/2023 00:14,NaN,Casual Member
20148785,722,7399,01/01/2023 00:01,Lower Jarvis / Queens Quay E,7533.0,01/01/2023 00:13,Housey St / Dan Leckie Way,Casual Member
20148786,1054,7269,01/01/2023 00:02,Toronto Eaton Centre (Yonge St),7076.0,01/01/2023 00:20,York St / Queens Quay W,Annual Member
20148790,1329,7721,01/01/2023 00:04,NaN,7685.0,01/01/2023 00:26,NaN,Casual Member
20148791,1291,7721,01/01/2023 00:04,NaN,7685.0,01/01/2023 00:26,NaN,Casual Member


In [3]:
# Column types and missing values
print("Columns:", df_raw.columns.tolist())
print("\nMissing values:")
print(df_raw.isna().sum())
print(f"\nUnique start stations: {df_raw['Start Station Name'].nunique()}")
print(f"Unique end stations: {df_raw['End Station Name'].nunique()}")

Columns: ['Trip Duration', 'Start Station Id', 'Start Time', 'Start Station Name', 'End Station Id', 'End Time', 'End Station Name', 'User Type']

Missing values:
Trip Duration              0
Start Station Id           0
Start Time                 0
Start Station Name    595075
End Station Id          2944
End Time                   0
End Station Name      598563
User Type                  0
dtype: int64

Unique start stations: 593
Unique end stations: 592


Small note on missing station names: Around 10% of trips are missing Start/End Station Name. These rows still have valid Station IDs, durations, and timestamps, so we retain them for temporal analysis but they cannot be used for any station-level work.

### Feature Engineering
Parse timestamps, extract temporal components (hour, day, month), create Weekday/Weekend and Peak Hour flags, and convert Trip Duration to minutes.

Peak hours are defined as Morning (7:00 to 9:59) and Evening (15:00 to 19:59), while everything else in off peak.

In [4]:
from src.features import parse_timestamps, add_derived_features, reorder_columns

df = parse_timestamps(df_raw)
df = add_derived_features(df)
df = reorder_columns(df)

print("Columns after feature engineering:")
print(df.columns.tolist())
df.head()

Columns after feature engineering:
['Trip_Duration_Min', 'Trip_Duration_MMSS', 'Trip Duration', 'Start Station Id', 'Start Station Name', 'Start Time', 'Start Hour', 'Start Day', 'Start Month', 'Start Date', 'End Station Id', 'End Station Name', 'End Time', 'End Day', 'End Month', 'End Date', 'User Type', 'Weekday_Weekend', 'Peak_Hour']


,Trip_Duration_Min,Trip_Duration_MMSS,Trip Duration,Start Station Id,Start Station Name,Start Time,Start Hour,Start Day,Start Month,Start Date,End Station Id,End Station Name,End Time,End Day,End Month,End Date,User Type,Weekday_Weekend,Peak_Hour
Trip Id,,,,,,,,,,,,,,,,,,,
20148784,14.000000,14:00,840,7022,Simcoe St / Queen St W,00:00,0,1,1,2023-01-01,7703.0,NaN,00:14,1,1,2023-01-01,Casual Member,Weekend,Off Peak
20148785,12.033333,12:02,722,7399,Lower Jarvis / Queens Quay E,00:01,0,1,1,2023-01-01,7533.0,Housey St / Dan Leckie Way,00:13,1,1,2023-01-01,Casual Member,Weekend,Off Peak
20148786,17.566667,17:34,1054,7269,Toronto Eaton Centre (Yonge St),00:02,0,1,1,2023-01-01,7076.0,York St / Queens Quay W,00:20,1,1,2023-01-01,Annual Member,Weekend,Off Peak
20148790,22.150000,22:09,1329,7721,NaN,00:04,0,1,1,2023-01-01,7685.0,NaN,00:26,1,1,2023-01-01,Casual Member,Weekend,Off Peak
20148791,21.516667,21:31,1291,7721,NaN,00:04,0,1,1,2023-01-01,7685.0,NaN,00:26,1,1,2023-01-01,Casual Member,Weekend,Off Peak


### Outlier Removal (IQR Method)

Trip Duration has extreme right skew (skewness ~ 7.4, kurtosis ~ 99), so z-score outlier detection is unreliable. We use the IQR method instead, which is robust to skewed distributions.

We also drop trips with duration below 0 (docking errors).

In [5]:
from src.features import remove_outliers_iqr

# Pre-removal stats
print(f"Before: {len(df):,} rows")
print(f"Skewness: {df['Trip_Duration_Min'].skew():.2f}")
print(f"Kurtosis: {df['Trip_Duration_Min'].kurtosis():.2f}")
print(f"Mean: {df['Trip_Duration_Min'].mean():.2f} min")
print(f"Median: {df['Trip_Duration_Min'].median():.2f} min")
print()

df = remove_outliers_iqr(df, column='Trip_Duration_Min', factor=1.5)

Before: 5,713,141 rows
Skewness: 164.93
Kurtosis: 39544.70
Mean: 17.99 min
Median: 11.45 min

IQR outlier removal (Trip_Duration_Min): 364,384 rows removed (6.38%), 5,344,361 rows remaining.
  Bounds: [-10.42, 36.12] minutes


In [6]:
# Post-removal stats
print(f"After: {len(df):,} rows")
print(f"Skewness: {df['Trip_Duration_Min'].skew():.2f}")
print(f"Kurtosis: {df['Trip_Duration_Min'].kurtosis():.2f}")
print(f"Mean: {df['Trip_Duration_Min'].mean():.2f} min")
print(f"Median: {df['Trip_Duration_Min'].median():.2f} min")

After: 5,344,361 rows
Skewness: 0.91
Kurtosis: 0.26
Mean: 12.56 min
Median: 10.82 min


### Add Modelling Features

Binary encodings and log-transformed duration for the modeling notebook.

In [7]:
from src.features import add_binary_features, add_log_duration

df = add_binary_features(df)
df = add_log_duration(df)

df.sample(5)

,Trip_Duration_Min,Trip_Duration_MMSS,Trip Duration,Start Station Id,Start Station Name,Start Time,Start Hour,Start Day,Start Month,Start Date,...,End Day,End Month,End Date,User Type,Weekday_Weekend,Peak_Hour,Peak_Hour_Binary,Weekday_Binary,User_Type_Binary,Log_Trip_Duration
2922844,13.950000,13:57,837,7016,Bay St / Queens Quay W (Ferry Terminal),09:03,9,9,8,2023-08-09,...,9,8,2023-08-09,Casual Member,Weekday,Morning,1,1,0,2.704711
5228584,6.750000,6:45,405,7770,NaN,08:05,8,14,12,2023-12-14,...,14,12,2023-12-14,Casual Member,Weekday,Morning,1,1,0,2.047693
5015693,18.333333,18:20,1100,7066,Willcocks St / St. George St,15:13,15,23,11,2023-11-23,...,23,11,2023-11-23,Casual Member,Weekday,Evening,1,1,0,2.961831
4216650,10.650000,10:39,639,7712,NaN,07:21,7,4,10,2023-10-04,...,4,10,2023-10-04,Casual Member,Weekday,Morning,1,1,0,2.455306
3263371,3.250000,3:15,195,7686,NaN,08:59,8,24,8,2023-08-24,...,24,8,2023-08-24,Casual Member,Weekday,Morning,1,1,0,1.446919


### Export Processed Data

In [8]:
output_path = PROJECT_ROOT / "data" / "processed" / "ridership_2023_clean.csv"
df.to_csv(output_path, index=False)
print(f"Saved {len(df):,} rows to {output_path.name}")

Saved 5,344,361 rows to ridership_2023_clean.csv
